In [1]:
import pandas as pd
import os
from pathlib import Path
import re
from datetime import datetime
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import rasterio
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from torchmetrics.classification import MulticlassF1Score
from glob import glob
import torch
import random
import torch.nn.functional as F
import numpy as np
import torch.nn as nn

C:\Users\lolke\miniconda3\envs\cosmohack\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds_metadata = pd.read_csv('./bs/meta.csv')
ds_metadata

,chip_id,kind,fire_event_id,region,epsg,x_min,y_min,x_max,y_max,width,...,s1_date_pre,s1_date_post,valid_frac,cloud_frac,landcover_top,n_fire_px,burn_area_ha,sev1_px,sev2_px,sev3_px
0,BS_tr_000001,bs,FE00657,NaN,32638.0,491520.0,5376000.0,501760.0,5386240.0,512,...,2019-04-17,2019-04-29,0.9963,0.0037,NaN,NaN,454.80,11360.0,10.0,0.0
1,BS_tr_000002,bs,FE01044,NaN,32638.0,471040.0,5068800.0,481280.0,5079040.0,512,...,2019-06-04,2019-06-21,0.8195,0.1805,NaN,NaN,2865.92,40651.0,1699.0,29298.0
2,BS_tr_000003,bs,FE01122,NaN,32638.0,655360.0,5396480.0,665600.0,5406720.0,512,...,2019-06-16,2019-06-28,0.9921,0.0079,NaN,NaN,2486.96,8335.0,35772.0,18067.0
3,BS_tr_000004,bs,FE01130,NaN,32638.0,614400.0,5345280.0,624640.0,5355520.0,512,...,2019-06-16,2019-06-28,0.9747,0.0253,NaN,NaN,2306.20,18761.0,32880.0,6014.0
4,BS_tr_000005,bs,FE01642,NaN,32638.0,481280.0,5171200.0,491520.0,5181440.0,512,...,2019-07-27,2019-08-15,0.7130,0.2870,NaN,NaN,1371.24,19588.0,12764.0,1929.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219,BS_tr_000220,bs,FE30913,NaN,32637.0,491520.0,5294080.0,501760.0,5304320.0,512,...,2024-07-03,2024-07-15,0.5881,0.4119,NaN,NaN,330.64,1404.0,4624.0,2238.0
220,BS_tr_000221,bs,FE31014,NaN,32638.0,266240.0,5140480.0,276480.0,5150720.0,512,...,2024-07-05,2024-07-24,1.0000,0.0000,NaN,NaN,3812.76,12671.0,20198.0,62450.0
221,BS_tr_000222,bs,FE31115,NaN,32637.0,645120.0,5150720.0,655360.0,5160960.0,512,...,2024-07-10,2024-07-22,1.0000,0.0000,NaN,NaN,1099.40,7943.0,10689.0,8853.0
222,BS_tr_000223,bs,FE31240,NaN,32637.0,696320.0,5283840.0,706560.0,5294080.0,512,...,2024-07-17,2024-08-10,0.9776,0.0224,NaN,NaN,3124.36,54608.0,8888.0,14613.0


In [3]:
s1_pre_folder = './bs/sentinel1_pre/'
s1_post_folder = './bs/sentinel1_post/'
s2_pre_folder = './bs/sentinel2_pre/'
s2_post_folder = './bs/sentinel2_post/'
aux_folder = './bs/aux/'

In [4]:
import rasterio
import numpy as np

idx = 21
s1_pre_prefix = "_Sentinel-1_pre"
s1_post_prefix = "_Sentinel-1_post"
s2_pre_prefix = "_Sentinel-2_pre"
s2_post_prefix = "_Sentinel-2_post"
aux_prefix = "_AUX"
path = aux_folder + ds_metadata.iloc[idx]["chip_id"] + aux_prefix + ".tif"

with rasterio.open(path) as src:
    s1_pre = src.read().astype(np.float32)  # shape: (C, H, W)
    print("shape:", s1_pre.shape, "dtype:", s1_pre.dtype)
    print("min/max:", s1_pre.min(), s1_pre.max())

shape: (3, 512, 512) dtype: float32
min/max: 4.0 141.0


In [5]:
import numpy as np
import rasterio

path = "bs/aux/BS_tr_000003_AUX.tif"  # подставь реальный путь

with rasterio.open(path) as src:
    aux = src.read().astype(np.float32)
    print("shape:", aux.shape, "dtype в файле:", src.dtypes, "nodata:", src.nodata)

for c in range(aux.shape[0]):
    ch = aux[c]
    finite = ch[np.isfinite(ch)]
    n_unique = len(np.unique(finite))
    print(f"\n--- канал {c} ---")
    print(f"min={finite.min():.2f} max={finite.max():.2f} mean={finite.mean():.2f}")
    print(f"n_unique={n_unique}")
    if n_unique <= 20:
        vals, counts = np.unique(finite, return_counts=True)
        print("уникальные значения и их частоты:", dict(zip(vals.tolist(), counts.tolist())))
    else:
        print("перцентили [1,25,50,75,99]:", np.percentile(finite, [1, 25, 50, 75, 99]))

shape: (3, 512, 512) dtype в файле: ('int16', 'int16', 'int16') nodata: None

--- канал 0 ---
min=-13.00 max=4.00 mean=-5.17
n_unique=18
уникальные значения и их частоты: {-13.0: 27532, -12.0: 523, -11.0: 2766, -10.0: 6764, -9.0: 14710, -8.0: 11386, -7.0: 13435, -6.0: 21161, -5.0: 25662, -4.0: 28859, -3.0: 36360, -2.0: 39721, -1.0: 21101, 0.0: 9513, 1.0: 1501, 2.0: 836, 3.0: 312, 4.0: 2}

--- канал 1 ---
min=1.00 max=1.00 mean=1.00
n_unique=1
уникальные значения и их частоты: {1.0: 262144}

--- канал 2 ---
min=20.00 max=90.00 mean=34.33
n_unique=6
уникальные значения и их частоты: {20.0: 104, 30.0: 227218, 40.0: 58, 60.0: 31517, 80.0: 469, 90.0: 2778}


In [6]:
import random
import numpy as np
import rasterio
import torch
from torch.utils.data import Dataset

S2_SCALE = 10000.0
S1_SCALE = 100.0

S1_CLIP_RANGES = {"VV": (-25.0, 0.0), "VH": (-32.5, 0.0)}

S2_BAND_IDX = {
    "B2": 0, "B3": 1, "B4": 2, "B5": 3, "B6": 4,
    "B7": 5, "B8A": 6, "B11": 7, "B12": 8, "SCL": 9,
}
SCL_INVALID = {0, 1, 3, 8, 9, 10}

WORLDCOVER_CODES = [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100]
CODE_TO_CLASS_IDX = {code: i for i, code in enumerate(WORLDCOVER_CODES)}
N_LANDCOVER_CLASSES = len(WORLDCOVER_CODES)

FOLDERS = {
    "s1_pre": ("./bs/sentinel1_pre/", "_Sentinel-1_pre"),
    "s1_post": ("./bs/sentinel1_post/", "_Sentinel-1_post"),
    "s2_pre": ("./bs/sentinel2_pre/", "_Sentinel-2_pre"),
    "s2_post": ("./bs/sentinel2_post/", "_Sentinel-2_post"),
    "aux": ("./bs/aux/", "_AUX"),
    "mask": ("./bs/masks/", "_MASK"),  # уточни реальную папку/суффикс маски у себя
}


def build_path(chip_id, kind):
    folder, suffix = FOLDERS[kind]
    return folder + chip_id + suffix + ".tif"


def load_tif(path):
    with rasterio.open(path) as src:
        return src.read().astype(np.float32)


def normalize_s1_db_scaled(arr, band_indices={"VV": 0, "VH": 1}, scale=S1_SCALE):
    h, w = arr.shape[1], arr.shape[2]
    out = np.zeros((2, h, w), dtype=np.float32)
    for out_i, band in enumerate(("VV", "VH")):
        src_i = band_indices[band]
        band_db = arr[src_i] / scale
        lo, hi = S1_CLIP_RANGES[band]
        band_db = np.clip(band_db, lo, hi)
        out[out_i] = (band_db - lo) / (hi - lo)
    return out


def process_s1_pair(s1_pre_raw, s1_post_raw):
    return normalize_s1_db_scaled(s1_pre_raw), normalize_s1_db_scaled(s1_post_raw)


def build_valid_mask_scl(scl_pre, scl_post):
    invalid_pre = np.isin(scl_pre, list(SCL_INVALID))
    invalid_post = np.isin(scl_post, list(SCL_INVALID))
    return (~(invalid_pre | invalid_post)).astype(np.float32)


def process_aux_3ch(aux_arr, elevation_range=(-15, 10)):
    """aux_arr: (3, H, W) -> [elevation, valid_mask, land_cover_class]"""
    elevation, aux_valid, land_cover = aux_arr
    elev_norm = np.clip((elevation - elevation_range[0]) /
                         (elevation_range[1] - elevation_range[0]), 0, 1).astype(np.float32)
    return elev_norm[None, ...], aux_valid.astype(np.float32)[None, ...], land_cover.astype(np.int64)


def landcover_to_onehot(land_cover_raw):
    h, w = land_cover_raw.shape
    onehot = np.zeros((N_LANDCOVER_CLASSES, h, w), dtype=np.float32)
    for code, class_idx in CODE_TO_CLASS_IDX.items():
        onehot[class_idx] = (land_cover_raw == code).astype(np.float32)
    return onehot


class CosmohackDataset(Dataset):
    def __init__(self, df, is_train=False):
        """df: срез ds_metadata (meta.csv) — должен содержать колонку chip_id."""
        self.df = df.reset_index(drop=True)
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def remap_mask(self, mask):
        return mask.astype(np.int64)  # BS-маска уже 0-3 согласно постановке кейса

    def apply_augmentations(self, x, mask):
        if random.random() > 0.5:
            x = np.flip(x, axis=2)
            mask = np.flip(mask, axis=1)
        if random.random() > 0.5:
            x = np.flip(x, axis=1)
            mask = np.flip(mask, axis=0)
        k = random.randint(0, 3)
        if k > 0:
            x = np.rot90(x, k, axes=(1, 2))
            mask = np.rot90(mask, k, axes=(0, 1))
        return np.ascontiguousarray(x), np.ascontiguousarray(mask)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        chip_id = row["chip_id"]

        s2_pre = load_tif(build_path(chip_id, "s2_pre")) / S2_SCALE
        s2_post = load_tif(build_path(chip_id, "s2_post")) / S2_SCALE

        idx_red, idx_nir, idx_swir1, idx_scl = (
            S2_BAND_IDX["B4"], S2_BAND_IDX["B8A"], S2_BAND_IDX["B12"], S2_BAND_IDX["SCL"]
        )

        pre_red, pre_nir, pre_swir1 = s2_pre[idx_red][None], s2_pre[idx_nir][None], s2_pre[idx_swir1][None]
        post_red, post_nir, post_swir1 = s2_post[idx_red][None], s2_post[idx_nir][None], s2_post[idx_swir1][None]
        scl_pre = (s2_pre[idx_scl] * S2_SCALE).round()
        scl_post = (s2_post[idx_scl] * S2_SCALE).round()

        pre_nbr1 = np.clip((pre_nir - pre_swir1) / (pre_nir + pre_swir1 + 1e-8), -1, 1)
        post_nbr1 = np.clip((post_nir - post_swir1) / (post_nir + post_swir1 + 1e-8), -1, 1)
        pre_ndvi = np.clip((pre_nir - pre_red) / (pre_nir + pre_red + 1e-8), -1, 1)
        post_ndvi = np.clip((post_nir - post_red) / (post_nir + post_red + 1e-8), -1, 1)
        dndvi = np.clip(pre_ndvi - post_ndvi, -2, 2)
        dnbr1 = np.clip(pre_nbr1 - post_nbr1, -2, 2)
        rdnbr = dnbr1 / (np.sqrt(np.abs(pre_nbr1)) + 1e-8)
        rdnbr = np.clip(rdnbr, -6.0, 6.0) / 6

        scl_valid_mask = build_valid_mask_scl(scl_pre, scl_post)[None, ...]

        s1_pre_raw = load_tif(build_path(chip_id, "s1_pre"))
        s1_post_raw = load_tif(build_path(chip_id, "s1_post"))
        pre_norm_s1, post_norm_s1 = process_s1_pair(s1_pre_raw, s1_post_raw)

        aux_raw = load_tif(build_path(chip_id, "aux"))
        elev_norm, aux_valid_mask, land_cover_raw = process_aux_3ch(aux_raw)
        land_cover_onehot = landcover_to_onehot(land_cover_raw)

        combined_valid_mask = scl_valid_mask * aux_valid_mask

        x = np.concatenate([
            pre_red, pre_nir, pre_swir1,
            post_red, post_nir, post_swir1,
            dndvi, dnbr1, rdnbr,
            pre_norm_s1, post_norm_s1,
            elev_norm,
            land_cover_onehot,
            combined_valid_mask,
        ], axis=0)

        mask_path = build_path(chip_id, "mask")
        mask = load_tif(mask_path).astype(np.int64)[0]
        mask = self.remap_mask(mask)

        if self.is_train:
            x, mask = self.apply_augmentations(x, mask)

        x_tensor = torch.from_numpy(x).float()
        mask_tensor = torch.from_numpy(mask).long()
        return x_tensor, mask_tensor

In [7]:
device = "cuda"

f1_metric = MulticlassF1Score(
    num_classes=4,
    average="macro",
    ignore_index=None
).to(device)

def compute_iou_per_class(logits, targets, num_classes=4):
    preds = logits.argmax(dim=1)
    ious = []
    for cls in range(num_classes):
        pred_cls = (preds == cls)
        target_cls = (targets == cls)
        intersection = (pred_cls & target_cls).sum().float()
        union = (pred_cls | target_cls).sum().float()
        ious.append((intersection / (union + 1e-8)).item())
    return ious

In [8]:
class ConvBlock(torch.nn.Module):
    def __init__(self, ch_in, ch_out):
        super().__init__()
        self.conv = nn.Sequential(
                                  nn.Conv2d(ch_in, ch_out,
                                            kernel_size=3, stride=1,
                                            padding=1, bias=True),
                                  nn.BatchNorm2d(ch_out),
                                  nn.ReLU(inplace=True),
                                  nn.Conv2d(ch_out, ch_out,
                                            kernel_size=3, stride=1,
                                            padding=1, bias=True),
                                  nn.BatchNorm2d(ch_out),
                                  nn.ReLU(inplace=True),
        )
        
    def forward(self, x):
        x = self.conv(x)
        return x

In [9]:
class UpConvBlock(torch.nn.Module):
    def __init__(self, ch_in, ch_out):
        super().__init__()
        self.up = nn.Sequential(
                                nn.Upsample(scale_factor=2),
                                nn.Conv2d(ch_in, ch_out,
                                         kernel_size=3,stride=1,
                                         padding=1, bias=True),
                                nn.BatchNorm2d(ch_out),
                                nn.ReLU(inplace=True),
        )
        
    def forward(self, x):
        x = x = self.up(x)
        return x

In [10]:
class AttentionBlock(torch.nn.Module):
    def __init__(self, f_g, f_l, f_int):
        super().__init__()
        
        self.w_g = nn.Sequential(
                                nn.Conv2d(f_g, f_int,
                                         kernel_size=1, stride=1,
                                         padding=0, bias=True),
                                nn.BatchNorm2d(f_int)
        )
        
        self.w_x = nn.Sequential(
                                nn.Conv2d(f_l, f_int,
                                         kernel_size=1, stride=1,
                                         padding=0, bias=True),
                                nn.BatchNorm2d(f_int)
        )
        
        self.psi = nn.Sequential(
                                nn.Conv2d(f_int, 1,
                                         kernel_size=1, stride=1,
                                         padding=0,  bias=True),
                                nn.BatchNorm2d(1),
                                nn.Sigmoid(),
        )
        
        self.relu = nn.ReLU(inplace=True)
        
    def forward(self, g, x):
        g1 = self.w_g(g)
        x1 = self.w_x(x)
        psi = self.relu(g1+x1)
        psi = self.psi(psi)
        
        return psi*x

In [11]:
class AttentionUNet(torch.nn.Module):
    def __init__(self, n_classes=4, in_channel=26, out_channel=4):
        super().__init__() 
        
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv1 = ConvBlock(ch_in=in_channel, ch_out=64)
        self.conv2 = ConvBlock(ch_in=64, ch_out=128)
        self.conv3 = ConvBlock(ch_in=128, ch_out=256)
        self.conv4 = ConvBlock(ch_in=256, ch_out=512)
        self.conv5 = ConvBlock(ch_in=512, ch_out=1024)
        
        self.up5 = UpConvBlock(ch_in=1024, ch_out=512)
        self.att5 = AttentionBlock(f_g=512, f_l=512, f_int=256)
        self.upconv5 = ConvBlock(ch_in=1024, ch_out=512)
        
        self.up4 = UpConvBlock(ch_in=512, ch_out=256)
        self.att4 = AttentionBlock(f_g=256, f_l=256, f_int=128)
        self.upconv4 = ConvBlock(ch_in=512, ch_out=256)
        
        self.up3 = UpConvBlock(ch_in=256, ch_out=128)
        self.att3 = AttentionBlock(f_g=128, f_l=128, f_int=64)
        self.upconv3 = ConvBlock(ch_in=256, ch_out=128)
        
        self.up2 = UpConvBlock(ch_in=128, ch_out=64)
        self.att2 = AttentionBlock(f_g=64, f_l=64, f_int=32)
        self.upconv2 = ConvBlock(ch_in=128, ch_out=64)
        
        self.conv_1x1 = nn.Conv2d(64, out_channel,
                                  kernel_size=1, stride=1, padding=0)
        
    def forward(self, x):
        # encoder
        x1 = self.conv1(x)
        
        x2 = self.maxpool(x1)
        x2 = self.conv2(x2)
        
        x3 = self.maxpool(x2)
        x3 = self.conv3(x3)
        
        x4 = self.maxpool(x3)
        x4 = self.conv4(x4)
        
        x5 = self.maxpool(x4)
        x5 = self.conv5(x5)
        
        # decoder + concat
        d5 = self.up5(x5)
        x4 = self.att5(g=d5, x=x4)
        d5 = torch.concat((x4, d5), dim=1)
        d5 = self.upconv5(d5)
        
        d4 = self.up4(d5)
        x3 = self.att4(g=d4, x=x3)
        d4 = torch.concat((x3, d4), dim=1)
        d4 = self.upconv4(d4)
        
        d3 = self.up3(d4)
        x2 = self.att3(g=d3, x=x2)
        d3 = torch.concat((x2, d3), dim=1)
        d3 = self.upconv3(d3)
        
        d2 = self.up2(d3)
        x1 = self.att2(g=d2, x=x1)
        d2 = torch.concat((x1, d2), dim=1)
        d2 = self.upconv2(d2)
        
        d1 = self.conv_1x1(d2)
        
        return d1

In [12]:
class OHEM(torch.nn.Module):
    def __init__(self, keep_ratio=0.7, ignore_index=255):
        """
        keep_ratio: доля (от 0.0 до 1.0) самых сложных пикселей батча.
                    0.7 означает, что градиент пойдет только по топ-70% сложных пикселей.
        """
        super().__init__()
        self.keep_ratio = keep_ratio
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        # logits: [B, C, H, W]
        # targets: [B, H, W]
        
        # 1. Попиксельный CrossEntropy без редукции
        loss = F.cross_entropy(
            logits, 
            targets, 
            reduction='none', 
            ignore_index=self.ignore_index
        ) # -> [B, H, W]

        # 2. Выпрямляем в 1D-вектор
        loss = loss.view(-1)
        
        # Если есть ignore_index, фильтруем нули/маску
        if self.ignore_index >= 0:
            valid_mask = (targets.view(-1) != self.ignore_index)
            loss = loss[valid_mask]

        num_pixels = loss.numel()
        keep_num = int(self.keep_ratio * num_pixels)

        if keep_num == 0 or keep_num >= num_pixels:
            return loss.mean()

        # 3. Отбираем топ самых больших ошибок через topk (быстрее полной сортировки sort)
        topk_loss, _ = torch.topk(loss, keep_num)
        
        # 4. Усредняем только по сложным пикселям
        return topk_loss.mean()

class DiceLoss(nn.Module):
    def __init__(self, num_classes, smooth=1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth

    def forward(self, logits, target):
        probs = F.softmax(logits, dim=1)
        target_oh = F.one_hot(target, self.num_classes).permute(0, 3, 1, 2).float()
        dims = (0, 2, 3)
        intersection = torch.sum(probs * target_oh, dims)
        cardinality = torch.sum(probs + target_oh, dims)
        dice = (2. * intersection + self.smooth) / (cardinality + self.smooth)
        return 1 - dice.mean()

class CombinedLoss(nn.Module):
    def __init__(self, class_weights, num_classes, ce_w=0.5, dice_w=0.5):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=class_weights)
        self.dice = DiceLoss(num_classes)
        self.ce_w, self.dice_w = ce_w, dice_w

    def forward(self, logits, target):
        return self.ce_w * self.ce(logits, target) + self.dice_w * self.dice(logits, target)

In [13]:
from torch.utils.data import WeightedRandomSampler

def calculate_sample_weights(df):   
    global_class_counts = np.zeros(4, dtype=np.int64)
    chip_stats = []
    
    for _, row in tqdm(df.iterrows(), total=len(df)):
        chip_id = row["chip_id"]
        mask_path = build_path(chip_id, "mask") 
        
        with rasterio.open(mask_path) as src:
            mask = src.read(1)
            
        counts = np.bincount(mask.ravel(), minlength=4)
        global_class_counts += counts
        chip_stats.append(counts)
    
    class_weights = 1.0 / (global_class_counts + 1)
    class_weights = class_weights / np.sum(class_weights)
    
    sample_weights = []
    for counts in chip_stats:
        weight = np.sum(counts * class_weights)
        sample_weights.append(weight)
        
    return sample_weights

In [ ]:
import numpy as np
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader

torch.backends.cudnn.benchmark = True
NUM_EPOCHS = 25

train_df, val_df = train_test_split(ds_metadata, test_size=0.1)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

train_df, test_df = train_test_split(train_df, test_size=0.12)
train_df = train_df.reset_index(drop=True)

train_sample_weights = calculate_sample_weights(train_df)

sampler = WeightedRandomSampler(
    weights=train_sample_weights,
    num_samples=len(train_sample_weights),
    replacement=True
)
pixel_counts = torch.tensor([52811255, 2362871, 2192271, 1353859], dtype=torch.float32)
total = pixel_counts.sum()
class_weights = total / (len(pixel_counts) * pixel_counts)
class_weights = class_weights / class_weights.sum() * len(pixel_counts)
class_weights = class_weights.to(device)

print(f"Train: {train_df.shape} \nVal: {val_df.shape} \nTest: {test_df.shape}")

train_ds = CosmohackDataset(train_df, is_train=True)
val_ds = CosmohackDataset(val_df)
test_ds = CosmohackDataset(test_df)

model = AttentionUNet().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9, weight_decay=0.1, nesterov=True)

criterion = OHEM().to(device)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=(amp_dtype == torch.float16))

train_loader = DataLoader(
    train_ds, batch_size=4,
    num_workers=0, pin_memory=True, sampler=sampler)

val_loader = DataLoader(
    val_ds, batch_size=4, shuffle=False, 
    num_workers=0, pin_memory=True)

test_loader = DataLoader(
    test_ds, batch_size=4, shuffle=False, 
    num_workers=0, pin_memory=True)

best_miou = 0.0

for epoch in tqdm(range(NUM_EPOCHS)):
    model.train()
    train_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(device, memory_format=torch.channels_last, non_blocking=True), masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            outputs = model(images)
            loss = criterion(outputs, masks)
        
        if amp_dtype == torch.bfloat16:
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        else:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
    model.eval()
    f1_metric.reset()
    val_loss = 0.0
    val_ious = []
    val_loss = 0.0
    total_intersection = torch.zeros(4, device=device)
    total_union = torch.zeros(4, device=device)
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device, memory_format=torch.channels_last, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                outputs = model(images)
                val_loss += criterion(outputs, masks).item()
            preds = outputs.argmax(dim=1)
            f1_metric.update(outputs, masks.long())

            for c in range(4):
                p_c = (preds == c)
                m_c = (masks == c)
                total_intersection[c] += (p_c & m_c).sum()
                total_union[c] += (p_c | m_c).sum()
            
            ious = compute_iou_per_class(outputs, masks)
            val_ious.append(ious)
    mean_ious = (total_intersection / (total_union + 1e-7)).cpu().numpy()
    f1_epoch = f1_metric.compute().item()
    val_loss /= len(val_loader)
    scheduler.step()
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss:.4f} | F1-score: {f1_epoch}\niou: {mean_ious.round(4)}, miou={mean_ious.mean()}")
    current_miou = mean_ious[1:].mean()
    if current_miou > best_miou:
        best_miou = current_miou
        torch.save(model.state_dict(), f"best_aunet_fold_cosmohack_sgd_sampler.pt")
        print(f"Чекпоинт сохранен: target mIoU = {best_miou:.4f}")

100%|██████████| 176/176 [00:00<00:00, 265.29it/s]


Train: (176, 26) 
Val: (23, 26) 
Test: (25, 26)


  4%|▍         | 1/25 [01:26<34:25, 86.08s/it]

Epoch 1/25 | Train Loss: 1.0544 | Val Loss: 0.6496 | F1-score: 0.2389591485261917
iou: [0.901 0.    0.    0.004], miou=0.22624625265598297
Чекпоинт сохранен: target mIoU = 0.0013


  8%|▊         | 2/25 [02:11<23:47, 62.04s/it]

Epoch 2/25 | Train Loss: 0.6618 | Val Loss: 0.4966 | F1-score: 0.47517627477645874
iou: [0.9272 0.     0.325  0.2886], miou=0.3851981461048126
Чекпоинт сохранен: target mIoU = 0.2045


In [18]:
import numpy as np
import rasterio
import glob

def load_mask(path):
    with rasterio.open(path) as src:
        return src.read().astype(np.int64)[0]

def class_distribution(mask_paths, n_classes=4):
    counts = np.zeros(n_classes, dtype=np.int64)
    for p in mask_paths:
        mask = load_mask(p)
        for c in range(n_classes):
            counts[c] += (mask == c).sum()
    return counts

mask_paths = glob.glob("./bs/masks/*.tif")
counts = class_distribution(mask_paths)
shares = counts / counts.sum()

for c in range(len(counts)):
    print(f"class {c}: {counts[c]} px ({shares[c]:.4%})")

class 0: 52811255 px (89.9370%)
class 1: 2362871 px (4.0239%)
class 2: 2192271 px (3.7334%)
class 3: 1353859 px (2.3056%)
